## Config & Imports
Point `PREP_ARTIFACT_DIR` to the output dataset from notebook 07.


In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import json
import math
import multiprocessing
import os
import random
import time

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.signal import savgol_filter
from scipy.spatial import cKDTree
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

try:
    import lightgbm as lgb
except Exception as e:
    lgb = None
    print("LightGBM unavailable:", repr(e))

try:
    from catboost import CatBoostRegressor, Pool
except Exception as e:
    CatBoostRegressor = None
    Pool = None
    print("CatBoost unavailable:", repr(e))

try:
    import optuna
except Exception as e:
    optuna = None
    print("Optuna unavailable; selection will use a deterministic grid:", repr(e))

try:
    from numba import njit, prange
except Exception as e:
    print("Numba unavailable; using Python fallback decorators. Full feature build will be slow:", repr(e))
    def njit(*args, **kwargs):
        if args and callable(args[0]):
            return args[0]
        return lambda f: f
    prange = range

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
PREP_ARTIFACT_CANDIDATES = [
    Path("/kaggle/input/notebooks/baopv05/rogii-eda-features"),
    Path("./output_aw/model_artifacts"),
]
PREP_ARTIFACT_DIR = next((p for p in PREP_ARTIFACT_CANDIDATES if (p / "train_df.joblib").exists()), PREP_ARTIFACT_CANDIDATES[0])  # TODO
OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./output_aw_catboost")
ARTIFACT_DIR = OUT_DIR 
MODEL_DIR = ARTIFACT_DIR / "models"
OOF_DIR = ARTIFACT_DIR / "oof"
SCORE_DIR = ARTIFACT_DIR / "scores"
for d in [ARTIFACT_DIR, MODEL_DIR, OOF_DIR, SCORE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
USE_GPU = IS_KAGGLE
N_SPLITS = 5
print("PREP_ARTIFACT_DIR:", PREP_ARTIFACT_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)


## Load Train Matrix
Use the exact feature list from prepare.


In [ ]:
if CatBoostRegressor is None:
    raise ImportError("CatBoost is required for this notebook")
train_df = joblib.load(PREP_ARTIFACT_DIR / "train_df.joblib")
features = json.loads((PREP_ARTIFACT_DIR / "features.json").read_text(encoding="utf-8"))
X = train_df[features]
y = train_df["target"]
groups = train_df["well"]
print("X:", X.shape, "y:", y.shape, "groups:", groups.nunique())


## Train CatBoost Candidates
Three seeds match the reference notebook. Save one model list and OOF per candidate.


In [ ]:
cb_params_base = dict(
    iterations=8000, depth=7, l2_leaf_reg=2.0,
    min_data_in_leaf=15, border_count=254,
    loss_function="RMSE", od_type="Iter", od_wait=300, verbose=250 if USE_GPU else False,
)
if USE_GPU:
    cb_params_base.update(task_type="GPU", devices="0:1")

cb_params = [
    dict(learning_rate=0.025, random_seed=42, **cb_params_base),
    dict(learning_rate=0.020, random_seed=7, **cb_params_base),
    dict(learning_rate=0.030, random_seed=123, **cb_params_base),
]

def train_catboost(params, name):
    oof = np.zeros(len(train_df), np.float32)
    models = []
    fold_scores = []
    splits = list(GroupKFold(n_splits=N_SPLITS).split(X, y, groups=groups))
    print(f"Training {name}")
    for fold_idx, (train_idx, valid_idx) in enumerate(splits):
        model = CatBoostRegressor(**params)
        model.fit(
            Pool(X.iloc[train_idx].values, label=y.iloc[train_idx].values),
            eval_set=Pool(X.iloc[valid_idx].values, label=y.iloc[valid_idx].values),
            use_best_model=True,
        )
        models.append(model)
        oof[valid_idx] = model.predict(X.iloc[valid_idx].values).astype(np.float32)
        score = root_mean_squared_error(y.iloc[valid_idx], oof[valid_idx])
        fold_scores.append(float(score))
        print(f"{name} fold {fold_idx} RMSE: {score:.4f}")
        gc.collect()
    overall = float(root_mean_squared_error(y, oof))
    joblib.dump(models, MODEL_DIR / f"{name}.joblib", compress=3)
    np.save(OOF_DIR / f"{name}_oof.npy", oof)
    (SCORE_DIR / f"{name}_scores.json").write_text(json.dumps({
        "candidate": name,
        "rmse": overall,
        "fold_scores": fold_scores,
    }, indent=2), encoding="utf-8")
    print(f"{name} overall RMSE: {overall:.4f}")
    return name, overall

scores = []
for i, params in enumerate(cb_params, 1):
    scores.append(train_catboost(params.copy(), f"catboost-{i}"))
pd.DataFrame(scores, columns=["candidate", "rmse"]).to_csv(ARTIFACT_DIR / "catboost_scores.csv", index=False)
print("Saved CatBoost artifacts")
